In [ ]:
taxon_id_soup_dict = {}
with open('taxonid_list.txt') as taxon_id_list:
    for taxon_id in taxon_id_list:
        url = f"https://meta-virome.org/MetaData/{taxon_id}"
        headers = {
            "User-Agent": (
                "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                "AppleWebKit/537.36 (KHTML, like Gecko) "
                "Chrome/124.0.0.0 Safari/537.36"
            ),
            "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
            "Accept-Language": "en-US,en;q=0.9",
        }

        r = requests.get(url, headers=headers, timeout=30)
        r.raise_for_status()

        soup = BeautifulSoup(r.text, "lxml")
        taxon_id_soup_dict[taxon_id] = soup




In [8]:
import csv
import time
import requests
from bs4 import BeautifulSoup

# The fields you want, in the order you want them in the output
FIELDS = [
    "Study Name",
    "Ncbi Phylum",
    "Ncbi Class",
    "Ncbi Order",
    "Ncbi Family",
    "Ncbi Genus",
    "Ncbi Species",
    "Geographic Location",
    "Isolation Country",
    "Sample Collection Date",
]

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
}


def parse_metadata(soup: BeautifulSoup) -> dict:
    """
    Walk every row of every meta-data table and build a {label: value} dict.
    Only picks up rows that have exactly two <td> cells.
    """
    data = {}
    for table in soup.select("table.meta-data-table"):
        for tr in table.find_all("tr"):
            tds = tr.find_all("td")
            if len(tds) != 2:
                continue
            key = tds[0].get_text(strip=True)
            val = tds[1].get_text(" ", strip=True)  # join split text nodes
            if key and key not in data:  # first wins; avoids header rows
                data[key] = val
    return data


def scrape_one(taxon_id: str) -> dict:
    url = f"https://meta-virome.org/MetaData/{taxon_id}"
    r = requests.get(url, headers=HEADERS, timeout=30)
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "lxml")
    parsed = parse_metadata(soup)
    row = {"Taxon ID": taxon_id}
    for f in FIELDS:
        row[f] = parsed.get(f, "")
    return row


def main():
    with open("taxonid_list.txt") as f:
        taxon_ids = [line.strip() for line in f if line.strip()]

    with open("metadata_output.csv", "w", newline="", encoding="utf-8") as fout:
        writer = csv.DictWriter(fout, fieldnames=["Taxon ID"] + FIELDS)
        writer.writeheader()

        for i, taxon_id in enumerate(taxon_ids, 1):
            try:
                row = scrape_one(taxon_id)
                print(f"[{i}/{len(taxon_ids)}] OK   {taxon_id}")
            except Exception as e:
                row = {"Taxon ID": taxon_id, **{f: "" for f in FIELDS}}
                print(f"[{i}/{len(taxon_ids)}] FAIL {taxon_id}: {e}")

            writer.writerow(row)
            fout.flush()          # so partial progress is saved if it crashes
            time.sleep(0.5)       # be polite; tune or drop if the site allows more


if __name__ == "__main__":
    main()

[1/1457] OK   638275778
[2/1457] OK   638276196
[3/1457] OK   638276414
[4/1457] OK   638276540
[5/1457] OK   638277058
[6/1457] OK   638277059
[7/1457] OK   642973510
[8/1457] OK   643677411
[9/1457] OK   645044502
[10/1457] OK   645948464
[11/1457] OK   650627014
[12/1457] OK   2595698266
[13/1457] OK   2595698268
[14/1457] OK   2595698269
[15/1457] OK   2595698270
[16/1457] OK   2595698271
[17/1457] OK   2595698480
[18/1457] OK   2595698847
[19/1457] OK   2599185111
[20/1457] OK   2599185124
[21/1457] OK   2599185127
[22/1457] OK   2700988754
[23/1457] OK   2700988761
[24/1457] OK   2700988772
[25/1457] OK   2700988777
[26/1457] OK   2700989055
[27/1457] OK   2700989072
[28/1457] OK   2700989079
[29/1457] OK   2706795369
[30/1457] OK   2706795546
[31/1457] OK   2706795571
[32/1457] OK   2706795626
[33/1457] OK   2706795629
[34/1457] OK   2706795889
[35/1457] OK   2706796135
[36/1457] OK   2706796487
[37/1457] OK   2706796503
[38/1457] OK   2706796667
[39/1457] OK   2757320737
[40/14